# Nightingale: EXP-012's single scoring of the test split, on Kaggle

The code and configuration were frozen at the tag `v0.9-freeze` (2026-11-09). This notebook opens the
test split **once** (`docs/05` §2, `docs/08` EXP-012) and scores every system and ablation at 100%, 50%
and 25% of the history, seeds 42-46, with the verdicts under the reading in force (`--reduced-claims
25-only`, amendment 3b).

**What it does, in order, and what stops it:**
1. clones the repository and checks that the graph is the one every earlier experiment scored;
2. fetches DDXPlus's `test.csv` (the pinned snapshot the laptop built from) and builds the test parquet
   through the freeze gate (`build_ddxplus_chestpain.py --split test --open-test-split`);
3. draws the evaluation masks, which reads no label, and **stops unless both digests equal the two recorded
   in `docs/08` before this run** (`0136f67f1ca6542f`, `795acf2689f1415e`);
4. scores once (`scripts/evaluate_ablations.py --split test --confirm-single-opening`), at the weights
   EXP-006 chose (`docs/results/exp006_fusion.json`). The runner records the opening before it reads the
   first label.

**Nothing is trained or tuned, and nothing is re-run after the opening.** A second scoring needs a stated
reason (`--reopen-because`) and is disclosed. Keep the notebook private. The test split is saved nowhere in
the output: it stays in `/tmp`.

## How to run it (about 10 minutes of your time)

1. **Create → New Notebook → File → Import Notebook**: upload `notebooks/kaggle_exp012_test.ipynb` from the laptop.
2. **Add Input → Your Datasets → `nightingale-models-exp012`** (the one from the rehearsal; nothing new to upload).
3. **Session options:** Accelerator **None**, Internet **On**.
4. **Save Version → Save & Run All (Commit) → Save**, then close the browser. Expect 45 to 90 minutes.
5. When Kaggle emails you, open the latest version → **Output** → download **`exp012_test.zip`** and unzip
   it into `D:\PRJ-1\medical\data\interim\exp012_test\`. Tell Claude.

Run it **once**. If a step before the scoring fails (the digests, the gate), nothing has been opened and
it can be run again; if the version stops during the scoring, tell Claude before running anything again.

In [ ]:
COMMIT = "master"  # the freeze tag must be an ancestor of it, and nothing the freeze fixed may differ
EXPECT_TOKENS = "0136f67f1ca6542f"  # docs/08, EXP-012's note of 2026-11-09, recorded before this run
EXPECT_ASKED = "795acf2689f1415e"

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

REPO = Path("/tmp/nightingale")  # outside /kaggle/working, so no data ends up in the output
OUT = Path("/kaggle/working/exp012_test")
OUT.mkdir(parents=True, exist_ok=True)


def run(*command, **kwargs):
    print("$", " ".join(map(str, command)), flush=True)
    subprocess.run([str(c) for c in command], check=True, **kwargs)


if not REPO.exists():
    run("git", "clone", "-q", "https://github.com/HarshRohila02/nightingale.git", REPO)
run("git", "fetch", "-q", "--tags", cwd=REPO)
run("git", "checkout", "-q", COMMIT, cwd=REPO)
run("git", "log", "--oneline", "-1", cwd=REPO)
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True, check=True
).stdout.strip()
# The models were trained, and are scored, with the versions the laptop uses (their run.json files).
run(sys.executable, "-m", "pip", "install", "-q", "xgboost==2.1.4", "neo4j==5.28.6", "python-dotenv==1.2.3")

In [ ]:
# The models, from the private dataset added as input (found by folder name, zipped or not).
sys.path.insert(0, str(REPO / "scripts"))
sys.path.insert(0, str(REPO))
from report_deep import place_models

BUNDLES = ("nightingale_b0_b1", "nightingale_b1_asked", "nightingale_b1_seeds", "nightingale_b1_dl")
found = place_models(Path("/kaggle/input"), REPO / "models", BUNDLES)
missing = [b for b in BUNDLES if not (REPO / "models" / b).exists()]
assert not missing, f"not found in /kaggle/input: {missing}. Add Input -> Your Datasets -> nightingale-models-exp012"
needed = [
    "nightingale_b0_b1/b0_prevalence.json",
    "nightingale_b0_b1/b1_logreg.json",
    "nightingale_b0_b1/b1_xgboost.ubj",
    "nightingale_b1_asked/b1_aug/b1_logreg.json",
    "nightingale_b1_asked/b1_asked_aug/b1_xgboost.ubj",
    "nightingale_b1_seeds/b1/seed46/b1_xgboost.ubj",
    "nightingale_b1_seeds/b1_aug/seed46/b1_logreg.json",
    "nightingale_b1_seeds/b1_asked_aug/seed46/b1_xgboost.ubj",
    "nightingale_b1_dl/b1_dl_aug/seed46/b1_dl.json",
    "nightingale_b1_dl/b1_dl_asked_aug/seed46/b1_dl.json",
]
absent = [n for n in needed if not (REPO / "models" / n).exists()]
assert not absent, f"the models dataset lacks {absent}"
print("placed:", found)

In [ ]:
# DDXPlus at the snapshot the laptop uses (docs/11 §4). validate.csv is fetched only because the decoder
# reads the release files beside it; the test split is the one file this notebook is for.
from huggingface_hub import hf_hub_download

for name in ["release_evidences.json", "release_conditions.json", "validate.csv", "test.csv"]:
    hf_hub_download(
        "aai530-group6/ddxplus",
        name,
        repo_type="dataset",
        revision="2ad986acc1ec62fb4a94171acc43f4fdd5bfde53",
        local_dir=str(REPO / "data" / "raw" / "ddxplus"),
    )
# BODHI-S (CC-BY-NC-4.0, research use only): the two files the knowledge graph reads. They stay in /tmp.
bodhi = REPO / "data" / "raw" / "bodhi_s"
bodhi.mkdir(parents=True, exist_ok=True)
for name in ["triples.jsonl", "nl_facts.jsonl"]:
    got = hf_hub_download("ekacare/BODHI-S", f"data/{name}", repo_type="dataset", local_dir="/tmp/bodhi_download")
    shutil.copy(got, bodhi / name)
run(sys.executable, "scripts/decode_ddxplus.py", cwd=REPO)

In [ ]:
# The graph must be the laptop's: the same sources give the same fingerprint (the one Aura holds).
from src.medical_kg.cardiac_kg import build_cardiac_kg
from src.medical_kg.neo4j_store import kg_fingerprint

interim = REPO / "data" / "interim"
kg = build_cardiac_kg(
    interim / "ddxplus_chestpain_conditions.json", interim / "ddxplus_evidences.json", bodhi
)
fingerprint = kg_fingerprint(kg)[:12]
print("graph fingerprint:", fingerprint)
assert fingerprint == "de349201157a", "this is not the graph the earlier experiments scored: stop here"

In [ ]:
# The test parquet, through the freeze gate: it refuses unless the tag is an ancestor of HEAD, nothing the
# freeze fixed has changed, and the working tree is clean. Then the masks, which read no label.
run("git", "status", "--short", cwd=REPO)
run(sys.executable, "scripts/build_ddxplus_chestpain.py", "--split", "test", "--open-test-split", cwd=REPO)
drawn = subprocess.run(
    [sys.executable, "scripts/evaluate_ablations.py", "--split", "test", "--digest-only"],
    cwd=REPO, capture_output=True, text=True, check=True,
).stdout
print(drawn)
import re

tokens, asked = re.search(r"mask digest.*?: (\w{16})\s+asked-set digest.*?: (\w{16})", drawn).groups()
assert (tokens, asked) == (EXPECT_TOKENS, EXPECT_ASKED), (
    f"the masks give {tokens}, {asked}, not the digests recorded in docs/08 "
    f"({EXPECT_TOKENS}, {EXPECT_ASKED}): STOP, nothing has been opened"
)
print("both digests reproduce: nothing has been scored yet")

In [ ]:
# SCORING: the single opening. The runner records it before the first label is read.
started = time.time()
with open(OUT / "scoring.log", "w", encoding="utf-8") as handle:
    scored = subprocess.run(
        [sys.executable, "scripts/evaluate_ablations.py", "--split", "test",
         "--confirm-single-opening", "--expect-digests", f"{EXPECT_TOKENS},{EXPECT_ASKED}",
         "--alpha-file", str(REPO / "docs" / "results" / "exp006_fusion.json"),
         "--reduced-claims", "25-only",
         "--out", str(OUT / "exp012_test.json")],
        cwd=REPO, stdout=handle, stderr=subprocess.STDOUT,
    )
status = "done" if scored.returncode == 0 else f"FAILED (exit {scored.returncode})"
minutes = (time.time() - started) / 60
print(open(OUT / "scoring.log", encoding="utf-8").read()[-3500:])
print(f"scoring {status} after {minutes:.0f} min")

In [ ]:
# What finished, and the one file to download (Output tab: exp012_test.zip). No test row is saved.
import hashlib

import numpy
import pandas
import xgboost

marker = REPO / "data" / "interim" / "test_split_opened.json"
if marker.exists():
    shutil.copy(marker, OUT / "test_split_opened.json")
facts = {
    "commit": commit,
    "graph_fingerprint": fingerprint,
    "scoring": f"{status}, {minutes:.0f} min",
    "digests": {"tokens": tokens, "asked": asked},
    "versions": {"python": sys.version.split()[0], "numpy": numpy.__version__,
                 "pandas": pandas.__version__, "xgboost": xgboost.__version__},
}
result = OUT / "exp012_test.json"
if result.exists():
    data = json.loads(result.read_text(encoding="utf-8"))
    facts["valid_as_a_result"] = data.get("valid_as_a_result")
    facts["not_a_result_because"] = data.get("not_a_result_because")
    facts["reduced_claims_in_force"] = data.get("reduced_claims_in_force")
    facts["freeze"] = data.get("freeze")
    facts["sha256"] = {"exp012_test.json": hashlib.sha256(result.read_bytes()).hexdigest()[:16]}
(OUT / "STATUS.txt").write_text(json.dumps(facts, indent=1, ensure_ascii=False), encoding="utf-8")
print(json.dumps(facts, indent=1, ensure_ascii=False))
archive = shutil.make_archive("/kaggle/working/exp012_test", "zip", OUT)
print(archive, f"{os.path.getsize(archive) / 1e6:.1f} MB")